In [1]:
'''
Sentiment scoring (build multiple, then compare)

Lexicon-based: VADER (general-purpose, fast baseline), Loughran-McDonald (finance-specific, handles words like "liability" or "tax" that VADER misreads as negative)
Bag-of-words + ML: TF-IDF/CountVectorizer → feed into logistic regression as your simplest learned model
Transformer-based: FinBERT as your top-line model
'''

'\nSentiment scoring (build multiple, then compare)\n\nLexicon-based: VADER (general-purpose, fast baseline), Loughran-McDonald (finance-specific, handles words like "liability" or "tax" that VADER misreads as negative)\nBag-of-words + ML: TF-IDF/CountVectorizer → feed into logistic regression as your simplest learned model\nTransformer-based: FinBERT as your top-line model\n'

In [2]:
'''
Score	Description
Positive (pos)	Represents the proportion of text that conveys positive sentiment.
Negative (neg)	Represents the proportion of text that conveys negative sentiment.
Neutral (neu)	Represents the proportion of text that is emotionally neutral.
Compound	A normalized score between -1 and +1 that indicates the overall sentiment of the text.
'''

'\nScore\tDescription\nPositive (pos)\tRepresents the proportion of text that conveys positive sentiment.\nNegative (neg)\tRepresents the proportion of text that conveys negative sentiment.\nNeutral (neu)\tRepresents the proportion of text that is emotionally neutral.\nCompound\tA normalized score between -1 and +1 that indicates the overall sentiment of the text.\n'

# Load earnings call transcripts and score sentiment with VADER and Loughran-McDonald

This notebook loads every transcript under the `Transcripts` folder, extracts each prepared remarks section, and compares two lexicon-based sentiment methods:
- VADER: fast general-purpose baseline
- Loughran-McDonald: finance-focused lexicon for words like `liability`, `tax`, and `risk`

The goal is to build a reproducible sentiment table before moving to ML or transformer models.

In [3]:
from pathlib import Path

import pandas as pd

from src.lexicon_sentiment import (
    load_transcripts,
    score_loughran_mcdonald,
    score_vader,
)

# Use the project root so the notebook works from the repo root or from the src folder.
project_root = Path.cwd().resolve()
if (project_root / 'Transcripts').exists():
    transcripts_dir = project_root / 'Transcripts'
else:
    transcripts_dir = project_root.parent / 'Transcripts'

print(f'Loading transcripts from: {transcripts_dir}')
transcripts = load_transcripts(transcripts_dir)
print(f'Loaded {len(transcripts)} transcript rows')
transcripts.head()

Loading transcripts from: /Users/nok/NLP-sentiment-signal-on-earnings-news/Transcripts
Loaded 188 transcript rows


,ticker,filename,transcript_path,prepared_remarks,raw_text
0,AAPL,2016-Apr-26-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Transcr...
1,AAPL,2016-Jan-26-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Brief\n...
2,AAPL,2016-Jul-26-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Brief\n...
3,AAPL,2016-Oct-25-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Brief\n...
4,AAPL,2017-Aug-01-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Brief\n...


In [4]:
transcripts['vader_score'] = transcripts['prepared_remarks'].apply(score_vader)
transcripts['lm_score'] = transcripts['prepared_remarks'].apply(score_loughran_mcdonald)

transcripts['vader_compound'] = transcripts['vader_score'].apply(lambda x: x.get('compound', 0.0))
transcripts['vader_pos'] = transcripts['vader_score'].apply(lambda x: x.get('pos', 0.0))
transcripts['vader_neg'] = transcripts['vader_score'].apply(lambda x: x.get('neg', 0.0))
transcripts['lm_compound'] = transcripts['lm_score'].apply(lambda x: x.get('compound', 0.0))
transcripts['lm_positive'] = transcripts['lm_score'].apply(lambda x: x.get('positive', 0.0))
transcripts['lm_negative'] = transcripts['lm_score'].apply(lambda x: x.get('negative', 0.0))

transcripts[['ticker', 'filename', 'vader_compound', 'lm_compound']].head(10)

,ticker,filename,vader_compound,lm_compound
0,AAPL,2016-Apr-26-AAPL.txt,0.0000,0.000000
1,AAPL,2016-Jan-26-AAPL.txt,0.9998,0.679012
2,AAPL,2016-Jul-26-AAPL.txt,0.9998,0.787234
3,AAPL,2016-Oct-25-AAPL.txt,0.9999,0.636364
4,AAPL,2017-Aug-01-AAPL.txt,0.9998,0.947368
5,AAPL,2017-Jan-31-AAPL.txt,0.9997,0.914286
6,AAPL,2017-May-02-AAPL.txt,0.9998,0.902439
7,AAPL,2017-Nov-02-AAPL.txt,0.9998,0.971429
8,AAPL,2018-Feb-01-AAPL.txt,0.9998,0.835294
9,AAPL,2018-Jul-31-AAPL.txt,0.9999,0.830986


In [5]:
summary = (
    transcripts.groupby('ticker')
    .agg(
        vader_mean=('vader_compound', 'mean'),
        lm_mean=('lm_compound', 'mean'),
        transcripts_count=('filename', 'count')
    )
    .sort_values('vader_mean', ascending=False)
)

summary.head(10)

,vader_mean,lm_mean,transcripts_count
ticker,,,
AAPL,0.894595,0.710703,19
ASML,0.137868,0.052632,19
AMD,0.000000,0.000000,19
AMZN,0.000000,0.000000,19
CSCO,0.000000,0.000000,19
GOOGL,0.000000,0.000000,19
INTC,0.000000,0.000000,19
MSFT,0.000000,0.000000,19
MU,0.000000,0.000000,17


In [6]:
# Save the scored output for downstream modeling or analysis
output_path = project_root / 'transcript_lexicon_scores.csv'
transcripts.to_csv(output_path, index=False)
print(f'Saved scored results to: {output_path}')

# Quick check of the result file
pd.read_csv(output_path).head()

Saved scored results to: /Users/nok/NLP-sentiment-signal-on-earnings-news/transcript_lexicon_scores.csv


,ticker,filename,transcript_path,prepared_remarks,raw_text,vader_score,lm_score,vader_compound,vader_pos,vader_neg,lm_compound,lm_positive,lm_negative
0,AAPL,2016-Apr-26-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Transcr...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...","{'positive': 0.0, 'negative': 0.0, 'neutral': ...",0.0000,0.000,0.000,0.000000,0.0,0.0
1,AAPL,2016-Jan-26-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Brief\n...,"{'neg': 0.018, 'neu': 0.839, 'pos': 0.144, 'co...","{'positive': 68.0, 'negative': 13.0, 'neutral'...",0.9998,0.144,0.018,0.679012,68.0,13.0
2,AAPL,2016-Jul-26-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Brief\n...,"{'neg': 0.012, 'neu': 0.854, 'pos': 0.135, 'co...","{'positive': 42.0, 'negative': 5.0, 'neutral':...",0.9998,0.135,0.012,0.787234,42.0,5.0
3,AAPL,2016-Oct-25-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Brief\n...,"{'neg': 0.014, 'neu': 0.821, 'pos': 0.165, 'co...","{'positive': 45.0, 'negative': 10.0, 'neutral'...",0.9999,0.165,0.014,0.636364,45.0,10.0
4,AAPL,2017-Aug-01-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Brief\n...,"{'neg': 0.018, 'neu': 0.839, 'pos': 0.144, 'co...","{'positive': 37.0, 'negative': 1.0, 'neutral':...",0.9998,0.144,0.018,0.947368,37.0,1.0
